### openEO

The openEO python client is used to connect to EODC's openEO API. You need to create an account and authenticate when connecting via openEO python client. For more information, see: https://docs.openeo.cloud/join/free_trial.html 

In [1]:
import openeo
from openeo.processes import mean

conn = openeo.connect("https://openeo-dev.eodc.eu/openeo/1.2.0")
conn = conn.authenticate_oidc()
print(conn)

Authenticated using refresh token.
<Connection to 'https://openeo-dev.eodc.eu/openeo/1.2.0/' with OidcBearerAuth>


As soon as you are connected, you can start exploring collections and processes and select the "SENTINEL5P_DAILY_AUT" collection in your loading process. Set the spatial and temporal extent of interest. 

In [2]:
cube_co = conn.load_collection(
    'SENTINEL5P_DAILY_AUT', 
    spatial_extent = dict(west=15, east=17, south=47, north=49, crs="EPSG:4326"), 
    temporal_extent = ["2019-01-01", "2025-12-31"])

You can then use processes, such as "aggregate_temporal_period" to create a monthly mean. 

In [3]:
cube_co_monthly_mean = cube_co.aggregate_temporal_period(period="month", dimension="time", reducer=mean)

In [4]:
cube_co_mean = cube_co_monthly_mean.reduce_dimension(dimension="x", reducer=mean).reduce_dimension(dimension="y", reducer=mean)

In [5]:
cube_save = cube_co_mean.save_result(format="zarr")

You then need to save your results and select a data format you want to save the data to. Then you can trigger the processing on the openEO API side by using "create_job" and "start_job". 

In [6]:
job_cube = cube_save.create_job(title="Sentinel5P-CO").start_job()

Preflight process graph validation raised: [404] Load call not available for SENTINEL5P_DAILY_AUT


In [8]:
job_cube

<BatchJob job_id='f96a061b-0cb5-47aa-bfd5-2a00c3e275cc'>

Once the job status says "finished", you can get and download your results. 

In [9]:
results = job_cube.get_results()
metadata = results.get_metadata()
results

<JobResults for job 'f96a061b-0cb5-47aa-bfd5-2a00c3e275cc'>

In [10]:
results.download_files()

[PosixPath('/home/vhutter/eodc-examples/demos/S5P/openeo_result.zarr'),
 PosixPath('/home/vhutter/eodc-examples/demos/S5P/job-results.json')]

We can only download files, not folders, so the result is zipped. To make the file available, we rename it and then unzip it.

In [12]:
import os

os.rename("openeo_result.zarr", "openeo_result.zip")

In [14]:
from zipfile import ZipFile

with ZipFile("openeo_result.zip") as zip:
    zip.extractall("openeo_result.zarr")

Finally, we use the zarr python package to open and explore the results.

In [15]:
import zarr 

dz = zarr.open_group("openeo_result.zarr")
dz

<Group file://openeo_result.zarr>

In [16]:
dz.tree()

/
├── carbonmonoxide_total_column (1, 1, 84, 3) float32
├── carbonmonoxide_total_column_n (1, 1, 84, 3) float32
├── qa_threshold (3,) float64
├── spatial_ref () int64
├── time (84,) int64
├── x (1,) float64
└── y (1,) float64

In [17]:
dz["carbonmonoxide_total_column"].shape

(1, 1, 84, 3)

In [18]:
dz["carbonmonoxide_total_column"][0,0,:,0]

array([0.03315409, 0.03414611, 0.03548333, 0.03623674, 0.0317809 ,
       0.02991975, 0.02887013, 0.03048145, 0.03028627, 0.0279909 ,
       0.02965838, 0.03114623, 0.0333245 , 0.03402673, 0.03533068,
       0.03669268, 0.03271559, 0.02919515, 0.02702155, 0.02885978,
       0.03053453, 0.03181287, 0.03137226, 0.03318483, 0.03548514,
       0.03484718, 0.03639646, 0.03555874, 0.03158629, 0.02920233,
       0.0313652 , 0.03709849, 0.03499237, 0.03258282, 0.03116717,
       0.03146774, 0.03317   , 0.03326112, 0.03527308, 0.03289822,
       0.03085526, 0.02761676, 0.02757056, 0.02813661, 0.02734851,
       0.02619535, 0.02752972, 0.03010155, 0.03019586, 0.03210232,
       0.03173329, 0.03214521, 0.03265199, 0.03163492, 0.03463256,
       0.03180658, 0.032269  , 0.03135214, 0.03250419, 0.03300713,
       0.03369486, 0.03366928, 0.03487796, 0.03311964, 0.03170405,
       0.02943   , 0.03237471, 0.03553003, 0.03418858, 0.03066337,
       0.03083515, 0.03233858, 0.0323187 , 0.0344116 , 0.03430

In [19]:
dz["carbonmonoxide_total_column"][0,0,:,1]

array([0.03361006, 0.03423112, 0.03585124, 0.03630733, 0.03188584,
       0.02995585, 0.02883539, 0.0305723 , 0.03033445, 0.02808741,
       0.02981262, 0.03130297, 0.03366391, 0.03425025, 0.03565124,
       0.03683275, 0.0328417 , 0.02923671, 0.02704956, 0.02882824,
       0.03042894, 0.03215661, 0.0313274 , 0.03333084, 0.03572065,
       0.03504151, 0.03654397, 0.03579092, 0.03169338, 0.0291973 ,
       0.0315684 , 0.03716949, 0.03518531, 0.03264518, 0.03117872,
       0.03157777, 0.03334663, 0.03333261, 0.03522504, 0.03305482,
       0.03081845, 0.02762518, 0.02748848, 0.02817321, 0.02748951,
       0.026283  , 0.02764888, 0.03024055, 0.03060853, 0.03225841,
       0.03216416, 0.0323666 , 0.03319248, 0.03179624, 0.03483516,
       0.03197198, 0.03237066, 0.03133652, 0.03271754, 0.03298988,
       0.03381752, 0.0341754 , 0.03541044, 0.03331097, 0.03171817,
       0.02952036, 0.03252534, 0.0355278 , 0.03419383, 0.03089393,
       0.03108827, 0.03248356, 0.0325716 , 0.03466127, 0.03433